# CP3 · Examen blanc de la partie III — solutions (notebook exécuté)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cemah2/workbookMaths/blob/main/checkpoints/partie_3/03_examen_solutions.ipynb)

L'enregistrement des réponses que vérifie le notebook de l'examen (cellules marquées `answer`, récoltées par `tools/build_answers.py`), calculées à partir des nombres du sujet ; les erreurs classiques reçoivent un message qui dit où chercher. Le corrigé détaillé et le barème sont dans `03_examen_corrige.md`.

In [1]:
# ⚙️ SETUP: run this cell first (on Colab or on your computer)
FAST_MODE = True  # True = quick version (a few seconds per cell); False = bigger experiments
SEED = 42

import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/cemah2/workbookMaths.git"
DRIVE_DIR = Path("/content/drive/MyDrive/workbookMaths")

try:
    import google.colab  # noqa: F401  (this module only exists on Colab)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    subprocess.run(["git", "config", "--global", "--add", "safe.directory", str(DRIVE_DIR)])
    if not (DRIVE_DIR / ".git").exists():
        subprocess.run(["git", "clone", REPO_URL, str(DRIVE_DIR)], check=True)
    else:
        # Get the new chapters. Your own work only lives in mon_travail/, which Claude never
        # touches, so "rebase + autostash" keeps it safe (even with local commits).
        identity = subprocess.run(["git", "-C", str(DRIVE_DIR), "config", "user.email"],
                                  capture_output=True).returncode == 0
        who = [] if identity else ["-c", "user.name=Workbook", "-c", "user.email=workbook@localhost"]
        pull = subprocess.run(["git", *who, "-C", str(DRIVE_DIR), "pull", "--rebase", "--autostash"],
                              capture_output=True, text=True)
        conflicts = subprocess.run(["git", "-C", str(DRIVE_DIR), "diff", "--name-only", "--diff-filter=U"],
                                   capture_output=True, text=True).stdout.split()
        if conflicts:  # a file changed outside mon_travail/ AND by Claude: git leaves conflict markers in it
            print("⚠️ Tes modifications de " + ", ".join(conflicts) + " (hors de mon_travail/) entrent en conflit "
                  "avec la nouvelle version ; elles sont gardées dans `git stash`. Pour reprendre la version du "
                  "dépôt : git restore --source=HEAD --staged --worktree <fichier> (00_setup/COLAB.md, « Problèmes "
                  "fréquents »).")
        elif pull.returncode != 0:
            print("⚠️ git pull a échoué (voir 00_setup/COLAB.md, « Problèmes fréquents ») :\n" + pull.stderr)
    ROOT = DRIVE_DIR
else:
    ROOT = Path(os.environ.get("WBM_ROOT", Path.cwd())).resolve()
    while not (ROOT / "src" / "wbm").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
    if not (ROOT / "src" / "wbm").exists():
        raise FileNotFoundError("Dépôt introuvable : ouvre ce notebook depuis le dossier de workbookMaths.")

os.environ["WBM_ROOT"] = str(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import wbm  # noqa: E402

cfg = wbm.setup(seed=SEED, fast=FAST_MODE)
FAST_MODE = cfg.fast

🧪 workbookMaths : environnement
   Plateforme : Local (Linux x86_64, 2 CPU)
   Python 3.13.18 | numpy 2.1.3 | scipy 1.16.3 | sympy 1.14.0 | matplotlib 3.10.0 | pandas 2.2.3 | scikit-learn 1.6.1
   Graine : 42 | FAST_MODE : True


## Les réponses vérifiées après l'examen

In [2]:
# The answers are computed from the numbers of the statements (01_examen_sujet.md)
import math
from fractions import Fraction
from math import comb


def dot(a, b):
    return sum(x * y for x, y in zip(a, b))


def transpose(a):
    return [list(col) for col in zip(*a)]


def matmul(a, b):
    """The product of two matrices given as lists of rows."""
    return [[dot(row, col) for col in zip(*b)] for row in a]


def whole(x):
    """x (a number, or a list of them) as int(s), after checking that it is a whole number: 6.000000000000001 -> 6."""
    if isinstance(x, (list, tuple)):
        return [whole(y) for y in x]
    assert abs(x - round(x)) < 1e-9, x
    return round(x)


def rank(m):
    """The rank of m (a list of rows), by Gaussian elimination on exact fractions."""
    rows, r = [[Fraction(x) for x in row] for row in m], 0
    for j in range(len(rows[0])):
        pivot = next((i for i in range(r, len(rows)) if rows[i][j] != 0), None)
        if pivot is None:
            continue
        rows[r], rows[pivot] = rows[pivot], rows[r]
        for i in range(r + 1, len(rows)):
            factor = rows[i][j] / rows[r][j]
            rows[i] = [a - factor * b for a, b in zip(rows[i], rows[r])]
        r += 1
    return r


# CP3.1: the 3 x 5 matrix A (columns c1, ..., c5), then V and W in R_2[X], a polynomial a + bX + cX² written (a, b, c)
A_1 = [[1, 0, 2, 3, -1], [0, 1, -3, 2, 4], [2, -1, 7, 4, -6]]
C_1 = transpose(A_1)
RANK_1 = rank(A_1)
COEF_1 = [A_1[0][2], A_1[1][2]]                # c3 = alpha c1 + beta c2, read on rows 1 and 2 (they begin with I)
COORD_1 = [C_1[4][0], C_1[4][1]]               # c5 = a c1 + b c2, read on rows 1 and 2 for the same reason
EQ_1 = [2, -1, -1]                             # z with A^T z = 0 and z1 = 2: z1 + 2 z3 = 0, z2 - z3 = 0
assert [COEF_1[0] * x + COEF_1[1] * y for x, y in zip(C_1[0], C_1[1])] == C_1[2]
assert [COORD_1[0] * x + COORD_1[1] * y for x, y in zip(C_1[0], C_1[1])] == C_1[4]
assert all(dot(EQ_1, c) == 0 for c in C_1)
SOLVABLE_1 = rank([row + [b] for row, b in zip(A_1, (3, 1, 4))]) == RANK_1


def at_1(t):
    """P(t) = a + b t + c t², as a row acting on (a, b, c)."""
    return [1, t, t * t]


COND_V_1 = [p - q for p, q in zip(at_1(2), at_1(-1))]    # P(2) - P(-1) = 0
DIM_V_1, DIM_W_1 = 3 - rank([COND_V_1]), 3 - rank([COND_V_1, at_1(0)])
P_1 = (7, -2, 2)
assert dot(COND_V_1, P_1) == 0                          # P is in V
H_1 = [P_1[0], P_1[2]]                                  # P = 7 * 1 + 2 * (X² - X)
assert [H_1[0], -H_1[1], H_1[1]] == list(P_1)

# CP3.2: Gram-Schmidt on (v1, v2), then the least squares by R x = Q^T y
V1_2, V2_2, Y_2 = (1, 1, 3, 5), (6, 0, 10, 0), (11, -6, 17, -4)
R11_2 = math.sqrt(dot(V1_2, V1_2))
Q1_2 = [x / R11_2 for x in V1_2]
R12_2 = dot(Q1_2, V2_2)
W_2 = [x - R12_2 * q for x, q in zip(V2_2, Q1_2)]       # v2 - r12 q1
R22_2 = math.sqrt(dot(W_2, W_2))
Q2_2 = [x / R22_2 for x in W_2]
QY_2 = [dot(Q1_2, Y_2), dot(Q2_2, Y_2)]
XH2_2 = QY_2[1] / R22_2                                 # back substitution: the last row first
XH_2 = [(QY_2[0] - R12_2 * XH2_2) / R11_2, XH2_2]
YH_2 = [QY_2[0] * a + QY_2[1] * b for a, b in zip(Q1_2, Q2_2)]     # Q Q^T y
assert all(abs(p - (XH_2[0] * a + XH_2[1] * b)) < 1e-9 for p, a, b in zip(YH_2, V1_2, V2_2))
RES_2 = [y - p for y, p in zip(Y_2, YH_2)]

# CP3.3: the eigenvalues of S from its trace and determinant; the Cholesky factor of B
S_3 = [[11, 15], [15, 27]]
TR_3, DET_3 = S_3[0][0] + S_3[1][1], S_3[0][0] * S_3[1][1] - S_3[0][1] * S_3[1][0]
EIG_3 = [TR_3 / 2 - math.sqrt(TR_3 ** 2 / 4 - DET_3), TR_3 / 2 + math.sqrt(TR_3 ** 2 / 4 - DET_3)]


def unit_eigenvector(m, lam):
    """The unit eigenvector of the symmetric 2 x 2 matrix m for lam, of positive first coordinate (first row of
    (m - lam I) v = 0)."""
    v = [m[0][1], lam - m[0][0]]
    n = math.hypot(*v)
    return [x / n for x in v] if v[0] > 0 else [-x / n for x in v]


B_3, X_3 = [[9, 3, -6], [3, 5, 0], [-6, 0, 9]], (1, -1, 1)


def cholesky(a):
    """The lower triangular L, of positive diagonal, such that a = L L^T."""
    n = len(a)
    low = [[0.0] * n for _ in range(n)]
    for i in range(n):
        for j in range(i + 1):
            s = a[i][j] - sum(low[i][k] * low[j][k] for k in range(j))
            low[i][j] = math.sqrt(s) if i == j else s / low[j][j]
    return low


L_3 = cholesky(B_3)
assert whole(matmul(L_3, transpose(L_3))) == B_3
LTX_3 = [dot(col, X_3) for col in transpose(L_3)]      # L^T x
LX_3 = [dot(row, X_3) for row in L_3]                   # L x
BOUND_3 = whole(math.sqrt(2 * 8))                       # [[2, a], [a, 8]]: 2 > 0 and 2 * 8 - a² > 0
INTERVAL_3 = f"]{-BOUND_3} ; {BOUND_3}["

# CP3.4: A = s1 u1 v1^T + s2 u2 v2^T, from the eigenvalues of A^T A
A_4 = [[9, 6], [2, -6]]
ATA_4, AAT_4 = matmul(transpose(A_4), A_4), matmul(A_4, transpose(A_4))
TR_4, DET_4 = ATA_4[0][0] + ATA_4[1][1], ATA_4[0][0] * ATA_4[1][1] - ATA_4[0][1] * ATA_4[1][0]
LAM_4 = [TR_4 / 2 + math.sqrt(TR_4 ** 2 / 4 - DET_4), TR_4 / 2 - math.sqrt(TR_4 ** 2 / 4 - DET_4)]
S_4 = [math.sqrt(lam) for lam in LAM_4]                 # s1 >= s2
V1_4, V2_4 = unit_eigenvector(ATA_4, LAM_4[0]), unit_eigenvector(ATA_4, LAM_4[1])
AV1_4 = [dot(row, V1_4) for row in A_4]
U1_4 = [x / S_4[0] for x in AV1_4]
U2_4 = [dot(row, V2_4) / S_4[1] for row in A_4]
A1_4 = [[S_4[0] * u * v for v in V1_4] for u in U1_4]
B_4 = [99, -22]                                         # = 11 sqrt(85) u1, so A^-1 b = sqrt(85) v1
SOL_4 = [math.sqrt(85) * v for v in V1_4]
assert all(abs(dot(row, SOL_4) - b) < 1e-9 for row, b in zip(A_4, B_4))
D_4 = [[a - b for a, b in zip(ra, rb)] for ra, rb in zip(A_4, A1_4)]    # A - A1
ERR_4 = math.sqrt(sum(x * x for row in D_4 for x in row))
FRO2_4 = sum(x * x for row in A_4 for x in row)
TRA_4, DETA_4 = A_4[0][0] + A_4[1][1], A_4[0][0] * A_4[1][1] - A_4[0][1] * A_4[1][0]
EIGA_4 = sorted((abs(TRA_4 / 2 + s * math.sqrt(TRA_4 ** 2 / 4 - DETA_4)) for s in (1, -1)), reverse=True)   # |λ| of A

# CP3.5: four small situations
T_5 = [3.2, 0.7, 6.1, 4.4, 2.9, 5.3]
N0_5, N1_5, N2_5 = 4, 10, 16
N_5 = N0_5 + N1_5 + N2_5
THETA_5 = Fraction(N0_5 + N1_5, 3 * N_5)                # (n0 + n1)/θ = 3 n2/(1 - 3θ)


def entropy(counts, log=math.log2):
    """The entropy of the frequencies of the counts (in bits by default)."""
    n = sum(counts)
    return -sum(c / n * log(c / n) for c in counts if c)


LAW_5 = (40, 25, 20, 15)                                # p = (0.4, 0.25, 0.2, 0.15), as counts out of 100
P_5 = [c / sum(LAW_5) for c in LAW_5]
KL_UNIFORM_5 = sum(p * math.log2(p / 0.25) for p in P_5)          # KL(p ‖ u) = log2(4) - H(p)
assert abs(KL_UNIFORM_5 - (2 - entropy(LAW_5))) < 1e-12
KL_REVERSED_5 = sum(0.25 * math.log2(0.25 / p) for p in P_5)      # KL(u ‖ p)
GINI_SQUARES_5 = sum(p ** 2 for p in P_5)
Z_5, LABEL_5 = [1.3, -1.2, -0.5, 2.2, 0.9], [1, 0, 1, 1, 0]
LOSSES_5 = [math.log(1 + math.exp(z)) - y * z for z, y in zip(Z_5, LABEL_5)]
LOGLOSS_5 = sum(LOSSES_5) / len(LOSSES_5)
CONSTANT_5 = (sum(LABEL_5), len(LABEL_5) - sum(LABEL_5))    # the best constant q is the frequency of the 1s

# CP3.6 (1): f = x + 2y on x²/8 + y²/2 = 1; ∇f + λ∇g = 0 gives x = -8/(2λ), y = -2 * 2/(2λ), then λ² from the constraint
C_6, AX_6, AY_6 = (1, 2), 8, 2
LAMS_6 = [s * math.sqrt(C_6[0] ** 2 * AX_6 + C_6[1] ** 2 * AY_6) / 2 for s in (1, -1)]
CANDIDATES_6 = [(-C_6[0] * AX_6 / (2 * lam), -C_6[1] * AY_6 / (2 * lam), lam) for lam in LAMS_6]
assert all(abs(x * x / AX_6 + y * y / AY_6 - 1) < 1e-12 for x, y, _ in CANDIDATES_6)
X_6, Y_6, LAM_6 = max(CANDIDATES_6, key=lambda c: dot(C_6, c[:2]))
XMIN_6, YMIN_6, _ = min(CANDIDATES_6, key=lambda c: dot(C_6, c[:2]))

# CP3.6 (2): the soft-margin SVM on three examples
W_6, B_6, CSVM_6 = (2, 1), -3, 1
DATA_6 = [((1, 3), 1), ((2, 0), -1), ((1, 0.5), 1)]


def hinge_6(data):
    return sum(max(0, 1 - y * (dot(W_6, x) + B_6)) for x, y in data)


HINGE_6 = hinge_6(DATA_6)
J_6 = dot(W_6, W_6) / 2 + CSVM_6 * HINGE_6
ACTIVE_6 = [(x, y) for x, y in DATA_6 if y * (dot(W_6, x) + B_6) < 1]   # the examples of margin below 1


def subgradient_6(examples, reg=1, sign=-1):
    """[g_w1, g_w2, g_b]: reg * w (the gradient of ½‖w‖²) + sign * C * y * (x1, x2, 1), summed over the examples."""
    g = [reg * W_6[0], reg * W_6[1], 0]
    for x, y in examples:
        for k, v in enumerate((*x, 1)):
            g[k] += sign * CSVM_6 * y * v
    return g


# CP3.6 (3): the Gram matrix of k(x, z) = (x + z)² on 1, 2, 3
PTS_6, VEC_6 = (1, 2, 3), (3, -2, 0)
K_6 = [[(x + z) ** 2 for z in PTS_6] for x in PTS_6]
VKV_6 = sum(VEC_6[i] * VEC_6[j] * K_6[i][j] for i in range(3) for j in range(3))
VKV_ONCE_6 = sum(VEC_6[i] * VEC_6[j] * K_6[i][j] for i in range(3) for j in range(i, 3))   # crossed terms once

# CP3.7: X of law B(18, 0.06), the Chernoff exponent and the Hoeffding bound of P(X >= 5); a limit; Hedge
N_7, P_7, K_7 = 18, 0.06, 5
A_7 = K_7 / N_7


def pmf_7(k):
    return comb(N_7, k) * P_7 ** k * (1 - P_7) ** (N_7 - k)


def kl(a, p):
    """KL(a ‖ p) between two Bernoulli laws, in nats."""
    return a * math.log(a / p) + (1 - a) * math.log((1 - a) / (1 - p))


TAIL_7 = sum(pmf_7(k) for k in range(K_7, N_7 + 1))
X_7, M_7 = -2.4, 45
D_7, LIMIT_7 = (1 + X_7 / M_7) ** M_7, math.exp(X_7)
GAP_7 = M_7 * math.log(1 + X_7 / M_7) - X_7                       # n ln(1 + x/n) - x
ETA_7, L_7 = 0.7, [3, 1, 2, 2]


def weights_7(exponents):
    """The weights proportional to exp(exponents), normalised to sum 1."""
    e = [math.exp(s) for s in exponents]
    return [x / sum(e) for x in e]


# CP3.10: six examples, three axes (the rows of U, divided by 21), the regression on the scores
X_10 = [[17, 13, 6], [17, 1, 12], [21, 6, 19], [7, 3, 24], [7, 15, 18], [3, 10, 11]]
Y_10 = [18, 18, 27, 20, 24, 13]
U_10 = [[19, -4, -8], [4, -13, 16], [8, 16, 11]]
assert all(dot(u, v) == (441 if i == j else 0) for i, u in enumerate(U_10) for j, v in enumerate(U_10))
N_10 = len(X_10)
MEAN_10 = [sum(col) / N_10 for col in zip(*X_10)]
YBAR_10 = sum(Y_10) / N_10


def scores_10(x, center=True):
    """[t1, t2, t3] of x: u_j · (x - mean), or u_j · x when center is False."""
    d = [a - m for a, m in zip(x, MEAN_10)] if center else x
    return [dot(u, d) / 21 for u in U_10]


T_10 = [scores_10(x) for x in X_10]
assert whole(T_10[1:]) == [[7, 3, -5], [7, 6, 4], [-7, 9, -1], [-7, -3, 5], [-7, -6, -4]]   # the table of the statement
SS_10 = [sum(t[j] ** 2 for t in T_10) for j in range(3)]                       # Σ t_j²
TY_10 = [sum(t[j] * (y - YBAR_10) for t, y in zip(T_10, Y_10)) for j in range(3)]
W_10 = [ty / ss for ty, ss in zip(TY_10, SS_10)]                                # the normal equations are diagonal
SCT_10 = sum((y - YBAR_10) ** 2 for y in Y_10)
R2_10 = [sum(W_10[j] * TY_10[j] for j in range(k)) / SCT_10 for k in (1, 2, 3)]   # with 1, 2 and 3 axes
NEW_10 = (26, 5, 13)


def predict_10(x, k, center=True):
    t = scores_10(x, center)
    return YBAR_10 + sum(W_10[j] * t[j] for j in range(k))

**CP3.1 — Indépendance, base, rang et noyau**

In [3]:
wbm.record("CP3.1a", RANK_1, mistakes={"le rang compte les lignes (ou les colonnes) indépendantes : une ligne est-elle une combinaison des autres ?": 3, "le rang ne dépasse ni le nombre de lignes, ni celui des colonnes": 5})
wbm.record("CP3.1b", len(C_1) - RANK_1, mistakes={"théorème du rang : le rang plus la dimension du noyau donne le nombre de colonnes": {0, 1}, "c'est le rang ; le théorème du rang relie le rang et la dimension du noyau": RANK_1})
wbm.record("CP3.1c", COEF_1, mistakes={"vérifie la relation sur la première ligne de A": [-x for x in COEF_1], "l'ordre suit celui de l'énoncé : α multiplie c₁, β multiplie c₂": COEF_1[::-1]})
wbm.record("CP3.1d", COORD_1, mistakes={"l'ordre des coordonnées suit celui de la base (c₁, c₂)": COORD_1[::-1], "vérifie le signe sur la première ligne": [-x for x in COORD_1]})
wbm.record("CP3.1e", EQ_1, mistakes={"Aᵀz = 0 dit que z est orthogonal à chaque colonne : vérifie z · c₁ et z · c₂": [2, 1, -1], "l'énoncé fixe z₁ = 2 : ta première coordonnée le respecte-t-elle ?": [-2, 1, 1], "la première coordonnée de z est fixée par l'énoncé : relis-le": [1, -0.5, -0.5]})
wbm.record("CP3.1f", SOLVABLE_1, mistakes={"un vecteur b n'est atteint par A que s'il est orthogonal à tout z de ker Aᵀ": True})
wbm.record("CP3.1g", DIM_W_1, mistakes={"W ajoute une condition à celle qui définit V": DIM_V_1, "P(2) = P(−1) est une condition linéaire sur les coefficients, et ℝ₂[X] est de dimension 3": 3})
wbm.record("CP3.1h", H_1, mistakes={"le coefficient de X dans X² − X est −1": [H_1[0], -H_1[1]], "l'ordre des coordonnées suit celui de la base (1, X² − X)": H_1[::-1]})

WBM_ANSWER {"hash": "08e90c27c71d4d1407a3df4fb3436106ed7c02e7b6683ee38ca5e5a706b2c1f5", "id": "CP3.1a", "kind": "int", "magnitude": 0, "mistakes": {"0edc440fe8c606b00710821e67afdc6b739f9680673c00e84d07c7936eaa97ba": "le rang compte les lignes (ou les colonnes) indépendantes : une ligne est-elle une combinaison des autres ?", "63ac98d7450359598ee5a1ec2576af74ee887d7750999243081e32477986389e": "le rang ne dépasse ni le nombre de lignes, ni celui des colonnes"}, "sign": 1}
📝 Ex CP3.1a : réponse enregistrée (int).
WBM_ANSWER {"hash": "74811ea3bc3ab7d16f58b54490de127112013b6ee62f3a1392280d5c7116a9d9", "id": "CP3.1b", "kind": "int", "magnitude": 0, "mistakes": {"9c75198af371194120646ed4c409fcddc4ed20b48dabe2568506b35153ddc86d": "théorème du rang : le rang plus la dimension du noyau donne le nombre de colonnes", "fad075226b8f91a8980cc720a5ff5b10f416e9869ab0c2402443892a294f69bf": "c'est le rang ; le théorème du rang relie le rang et la dimension du noyau", "ff59c688d1c5cfb82177d0eab628d7542ace

**CP3.2 — Gram-Schmidt, projection et moindres carrés**

In [4]:
wbm.record("CP3.2a", whole(R11_2), mistakes={"la norme est la racine carrée de la somme des carrés": dot(V1_2, V1_2)})
wbm.record("CP3.2b", whole(R12_2), mistakes={"r₁₂ = q₁ · v₂, avec q₁ unitaire": dot(V1_2, V2_2), "r₁₂ est la coordonnée de v₂ sur q₁ : le produit scalaire q₁ · v₂": 1})
wbm.record("CP3.2c", whole(W_2), mistakes={"c'est q₂ ; la question demande le vecteur avant normalisation": Q2_2, "on retire r₁₂ q₁, où q₁ est v₁ divisé par sa norme": whole([x - R12_2 * v for x, v in zip(V2_2, V1_2)])})
wbm.record("CP3.2d", whole([[R11_2, R12_2], [0, R22_2]]), mistakes={"R est triangulaire supérieure": whole([[R11_2, 0], [R12_2, R22_2]]), "r₂₂ est une norme, pas un carré de norme": whole([[R11_2, R12_2], [0, R22_2 ** 2]])})
wbm.record("CP3.2e", whole(QY_2), mistakes={"c'est Xᵀy ; la question demande Qᵀy, avec les vecteurs orthonormés": [dot(V1_2, Y_2), dot(V2_2, Y_2)]})
wbm.record("CP3.2f", whole(YH_2), mistakes={"c'est le résidu y − ŷ ; la question demande la projection": whole(RES_2)})
wbm.record("CP3.2g", whole(XH_2), mistakes={"l'ordre des inconnues : x̂₁ multiplie la première colonne de R": whole(XH_2[::-1]), "R x̂ = Qᵀy se résout de bas en haut : la dernière ligne donne x̂₂, puis on remonte": whole([QY_2[0] / R11_2, QY_2[1] / R22_2])})
wbm.record("CP3.2h", float(whole(dot(RES_2, RES_2))), decimals=2, mistakes={"la question demande le carré de la distance": math.sqrt(dot(RES_2, RES_2)), "c'est ‖ŷ‖² ; la question porte sur le résidu y − ŷ": float(whole(dot(YH_2, YH_2)))})

WBM_ANSWER {"hash": "075e1d983f98c68c5d09a6e597c80c92ef8fb7e9505492bc2348ab7da82263f1", "id": "CP3.2a", "kind": "int", "magnitude": 0, "mistakes": {"eb8ec939b69252c25db76295be20849f76190cdd6b282f1cc7ec6e69d6c74285": "la norme est la racine carrée de la somme des carrés"}, "sign": 1}
📝 Ex CP3.2a : réponse enregistrée (int).
WBM_ANSWER {"hash": "a3250ac4f8bb880a248e67f5010946f91063b320b848899c39691d9d817f687c", "id": "CP3.2b", "kind": "int", "magnitude": 0, "mistakes": {"7a18d51cc9a9e80256def7c2e9226afe744c2103cde9ff87012b42da8994cd7e": "r₁₂ est la coordonnée de v₂ sur q₁ : le produit scalaire q₁ · v₂", "bc1eaf01d2898a287bc19f82e76311764e28fbfc58e1a8d2673264b2cff9857a": "r₁₂ = q₁ · v₂, avec q₁ unitaire"}, "sign": 1}
📝 Ex CP3.2b : réponse enregistrée (int).
WBM_ANSWER {"decimals": 0, "element_hashes": ["7b05c61304d89337689b5495ce3b4f4ea3d5934ccef1de51c904120480312fc1", "88623da798263c9ed7f49eb7a04fdf1ab538b1689e3787c82ad8e5d358af5508", "352a29ebba962a12897d44dc024a50cae9fe6fb8c89bab59d7b5

**CP3.3 — Valeurs propres, matrice définie positive et Cholesky**

In [5]:
wbm.record("CP3.3a", whole(EIG_3), mistakes={"la question demande l'ordre croissant": whole(EIG_3[::-1])})
wbm.record("CP3.3b", unit_eigenvector(S_3, EIG_3[1]), decimals=4, mistakes={"c'est un vecteur propre de la plus petite valeur propre": unit_eigenvector(S_3, EIG_3[0]), "le vecteur demandé est unitaire : divise-le par sa norme": [3, 5], "la question demande la première coordonnée positive": [-x for x in unit_eigenvector(S_3, EIG_3[1])]})
wbm.record("CP3.3c", 1 / EIG_3[0], decimals=4, mistakes={"S⁻¹ a pour valeurs propres les inverses de celles de S": EIG_3[1], "c'est la plus petite valeur propre de S⁻¹": 1 / EIG_3[1]})
wbm.record("CP3.3d", rank([[S_3[0][0] - 2, S_3[0][1]], [S_3[1][0], S_3[1][1] - 2]]), mistakes={"2 est valeur propre de S, donc S − 2I est singulière ; mais elle n'est pas nulle": {0, 2}})
wbm.record("CP3.3e", whole(L_3), mistakes={"L est triangulaire inférieure": whole(transpose(L_3))})
wbm.record("CP3.3f", whole(math.prod(L_3[i][i] for i in range(3)) ** 2), mistakes={"det B = det L × det Lᵀ": whole(math.prod(L_3[i][i] for i in range(3)))})
wbm.record("CP3.3g", dot(LTX_3, LTX_3), decimals=2, mistakes={"xᵀBx = xᵀLLᵀx = ‖Lᵀx‖² : c'est Lᵀ, pas L, qui s'applique à x": dot(LX_3, LX_3), "la question demande un carré de norme": math.sqrt(dot(LTX_3, LTX_3))})
wbm.record("CP3.3h", INTERVAL_3, kind="interval", mistakes={"définie positive demande des inégalités strictes": "[-4 ; 4]", "une inégalité sur un carré se résout avec une racine : que donne a² < 16 pour a ?": "]-16 ; 16[", "a peut être négatif : seul a² compte": "]0 ; 4["})

WBM_ANSWER {"decimals": 0, "element_hashes": ["581f8231e959fc88b68bc3934cea20a22443b9dc98ae9f23b5dfa0496239e341", "17eacf723b9b9308520b766402b878e0d0804e2efd52c5207a64984107e690ff"], "hash": "fd1556c79cdd45eff4a8141a6195210c16f5750c5b85223331a3c78078cf8509", "id": "CP3.3a", "integer": true, "kind": "array", "mistakes": {"bf7408a23f50dca4a168d3f8493bb2bcd181811f2bf945591840bf00a0b108c2": "la question demande l'ordre croissant"}, "shape": [2]}
📝 Ex CP3.3a : réponse enregistrée (array, 0 décimale).
WBM_ANSWER {"decimals": 4, "element_coarse": [["e2632e78c9045e1e4fafe292d5a12fce749b318eb2c0409e07fd78357dde2824"], ["5983476415110afae1375f4da1069d08b358b22ba0ff3b7985c04930bda50572"]], "element_coarse_any": [["e8c13701350b64058548c736b82e6423fee051863f4c544f62c9319db2662d88", "04d25fb24830fbb161639e9422d516ce1981ba90c8ab784d57a096d98a30bab8", "4375fa06d268a994746a227856666720040bb6ecc24cb473ac003199becc8211"], ["40d457bef0a5e22062ee358632ba64c462ce8bb2f21a9dd89d20bfe6f71ade3c", "f6f6eabc1af01

WBM_ANSWER {"hash": "697614e5050c4a1ece4286b5bb49f77fd5aa2ac34cdbf1ad9eddc6467350d44a", "id": "CP3.3h", "kind": "interval", "mistakes": {"0998045f58ed47b131e228b8ad45a7fd264d9b75fc84e1f673fb8cbc471c236f": "a peut être négatif : seul a² compte", "0f41ae967c08c3b126bb49f7e88faf4bda13ed7457d5ac76fc588ce8b2841186": "une inégalité sur un carré se résout avec une racine : que donne a² < 16 pour a ?", "4ad45fb591f11c375000f4a98a62eae0f0c564a89061f5ce0d9edb4cea3b896f": "définie positive demande des inégalités strictes"}}
📝 Ex CP3.3h : réponse enregistrée (interval).


**CP3.4 — SVD, rang faible et variance expliquée**

In [6]:
wbm.record("CP3.4a", ATA_4, mistakes={"c'est AAᵀ ; la question demande AᵀA": AAT_4})
wbm.record("CP3.4b", whole(S_4), mistakes={"les valeurs singulières sont les racines carrées des valeurs propres de AᵀA": whole(LAM_4), "la convention range les valeurs singulières dans l'ordre décroissant": whole(S_4[::-1]), "A n'est pas symétrique : ses valeurs singulières ne sont pas ses valeurs propres ; passe par AᵀA": EIGA_4})
wbm.record("CP3.4c", V1_4, decimals=3, mistakes={"c'est v₂ : v₁ est associé à la plus grande valeur propre de AᵀA": V2_4, "v₁ doit être unitaire : divise-le par sa norme": [7, 6]})
wbm.record("CP3.4d", U1_4, decimals=3, mistakes={"divise A v₁ par s₁": AV1_4, "c'est u₂ ; la question demande u₁ = A v₁ / s₁": U2_4})
wbm.record("CP3.4e", whole(SOL_4), mistakes={"A n'est pas orthogonale : son inverse n'est pas Aᵀ, il divise par les valeurs singulières": [dot(col, B_4) for col in transpose(A_4)], "la solution s'écrit avec v₁, pas avec u₁": whole([9, -2]), "A⁻¹ divise la composante sur u₁ par s₁": whole([11 * x for x in SOL_4])})
wbm.record("CP3.4f", whole(ERR_4), mistakes={"c'est le carré de l'erreur": whole(ERR_4 ** 2), "l'erreur vient de la valeur singulière retirée, pas de celle qui est gardée": whole(S_4[0])})
wbm.record("CP3.4g", S_4[0] ** 2 / FRO2_4, decimals=3, mistakes={"la part d'énergie se calcule avec les carrés des valeurs singulières": S_4[0] / sum(S_4), "c'est la part perdue ; la question demande la part gardée": 1 - S_4[0] ** 2 / FRO2_4})
wbm.record("CP3.4h", S_4[0] / S_4[1], decimals=3, mistakes={"c'est le conditionnement de AᵀA ; la question porte sur A": LAM_4[0] / LAM_4[1], "κ₂ divise la plus grande valeur singulière par la plus petite, pas l'inverse": S_4[1] / S_4[0], "A n'est pas symétrique : κ₂ se calcule avec les valeurs singulières, pas avec les valeurs propres": EIGA_4[0] / EIGA_4[1]})

WBM_ANSWER {"decimals": 0, "element_hashes": ["3ae878c6870fd49de7ec35c066eb1ede2621fddd2eb689bbe6436d1a4956194b", "91418ebcad9308f285dc811e372633fbdb07daa03d0bbe4a2c89804bb4ee03ed", "ae2632cc20a0764039224b21179a9cd375ddec184e83fba08f23a559c0f2be8b", "6bbd7f68111c7a16626c8cdb806537ffad1cee0846045384df388bda3cca5d13"], "hash": "cabd35f42ba9ed5bf79c0a6ba710aee802f26e523027b8773126f0852c97fccb", "id": "CP3.4a", "integer": true, "kind": "array", "mistakes": {"31192b71b1d80911791bef5f9a084e7f0b65046759bb1412cb77251e3526b69d": "c'est AAᵀ ; la question demande AᵀA"}, "shape": [2, 2]}
📝 Ex CP3.4a : réponse enregistrée (array, 0 décimale).
WBM_ANSWER {"decimals": 0, "element_hashes": ["3b255327d77b27b7593b8f4ea52c36ba7adbfa887db718c48a01e0d10a5b7a0e", "6e036af9b37eef18e6f32c62d74fb9b9f173de3ded518919608b7a5bd0acab18"], "hash": "e29eb8c0dd271eb9f2aad447884f005cd047e2d725741b75cf2dcc5c22927b17", "id": "CP3.4b", "integer": true, "kind": "array", "mistake_decimals": 2, "mistakes": {"1d802165caea520d

**CP3.5 — Maximum de vraisemblance, log-loss et entropie**

In [7]:
wbm.record("CP3.5a", max(T_5), decimals=1, mistakes={"la vraisemblance est nulle dès que θ est plus petit qu'une observation": sum(T_5) / len(T_5), "c'est l'estimateur des moments ; la question demande le maximum de vraisemblance": 2 * sum(T_5) / len(T_5)})
wbm.record("CP3.5b", THETA_5, kind="fraction", simplified=True, mistakes={"les observations 1 et 2 dépendent aussi de θ : la fréquence des 0 ne suffit pas": Fraction(N0_5, N_5), "la fréquence des 1 ne suffit pas non plus : les trois comptes dépendent de θ": Fraction(N1_5, 2 * N_5), "vérifie la dérivée de ln(1 − 3θ)": Fraction(N0_5 + N1_5, 3 * (N0_5 + N1_5) + N2_5), "c'est l'estimateur des moments ; la question demande le maximum de vraisemblance": Fraction(2 * N_5 - (N1_5 + 2 * N2_5), 4 * N_5)})
wbm.record("CP3.5c", entropy(LAW_5), decimals=3, mistakes={"en bits : le logarithme de base 2": {entropy(LAW_5, math.log), 1.319}})
wbm.record("CP3.5d", 1 - GINI_SQUARES_5, decimals=3, mistakes={"l'indice de Gini est 1 − Σ p²": GINI_SQUARES_5})
wbm.record("CP3.5e", KL_UNIFORM_5, decimals=3, mistakes={"la KL n'est pas symétrique : la question demande KL(p ‖ u)": KL_REVERSED_5, "en bits : le logarithme de base 2": KL_UNIFORM_5 * math.log(2)})
wbm.record("CP3.5f", LOGLOSS_5, decimals=3, mistakes={"la log-loss moyenne divise la somme par le nombre d'exemples": sum(LOSSES_5), "en nats : le logarithme népérien": LOGLOSS_5 / math.log(2), "le terme d'un exemple est s(z) − yz : as-tu retranché yz ?": sum(math.log(1 + math.exp(z)) for z in Z_5) / len(Z_5)})
wbm.record("CP3.5g", entropy(CONSTANT_5, math.log), decimals=3, mistakes={"la question demande des nats : le logarithme népérien": entropy(CONSTANT_5), "0,5 n'est pas la meilleure probabilité constante pour ces labels": math.log(2)})

WBM_ANSWER {"decimals": 1, "hash": "449cb9e2de22d6790d64669d1149b285596c777ffa6bf0f07b167ed2d45fe385", "hash_coarse": "0fef3ec86fe95857d6df1863fcf56b1cc78074385af6f8f1b1835ddb609cb70b", "id": "CP3.5a", "kind": "float", "magnitude": 0, "mistakes": {"a7fe905f457853c6fe8105d3efa3b7ea64a781fd880878ce2eb7c60d679a8255": "c'est l'estimateur des moments ; la question demande le maximum de vraisemblance", "b19d8c762f9dabe0b2a70d215f58437518259b3f66d53c42c295d24cf078ac42": "la vraisemblance est nulle dès que θ est plus petit qu'une observation"}, "sign": 1}
📝 Ex CP3.5a : réponse enregistrée (float, 1 décimale).
WBM_ANSWER {"hash": "1d58045418eb336afe6851ef1aa5725d9848dd317fdd7d3fd8190136b4d41fe3", "id": "CP3.5b", "kind": "fraction", "mistakes": {"3cead4bb671d848a26914c4575b5e47ffb0dce2301112cc46ca62fc22ec65acd": "la fréquence des 1 ne suffit pas non plus : les trois comptes dépendent de θ", "786da784550727f3e7cf0b0b15b5d99ce6ea00e34f7945adb2fbadd2de4e334c": "vérifie la dérivée de ln(1 − 3θ)", "8

**CP3.6 — Lagrange, hinge loss et noyau**

In [8]:
wbm.record("CP3.6a", whole([X_6, Y_6]), mistakes={"c'est le point du minimum de f sur l'ellipse": whole([XMIN_6, YMIN_6])})
wbm.record("CP3.6b", whole(dot(C_6, (X_6, Y_6))), mistakes={"c'est le minimum de f sur l'ellipse": whole(dot(C_6, (XMIN_6, YMIN_6)))})
wbm.record("CP3.6c", whole(LAM_6), mistakes={"avec la convention ∇f + λ∇g = 0, vérifie le signe": whole(-LAM_6), "vérifie ∇g : quelle est la dérivée de x²/8 ?": -4})
wbm.record("CP3.6d", HINGE_6, decimals=2, mistakes={"la marge d'un exemple est y (w · x + b) : quel est le label de P₂ ?": hinge_6([(x, abs(y)) for x, y in DATA_6])})
wbm.record("CP3.6e", J_6, decimals=2, mistakes={"le terme de régularisation est la moitié de ‖w‖²": dot(W_6, W_6) + CSVM_6 * HINGE_6, "n'oublie pas le terme ½‖w‖²": CSVM_6 * HINGE_6})
wbm.record("CP3.6f", subgradient_6(ACTIVE_6), decimals=2, mistakes={"n'oublie pas le gradient de ½‖w‖², qui est w": subgradient_6(ACTIVE_6, reg=0), "seuls les exemples de marge strictement inférieure à 1 ont un terme non nul": subgradient_6(DATA_6), "le gradient du terme d'un exemple actif est −y x, pas +y x": subgradient_6(ACTIVE_6, sign=1)})
wbm.record("CP3.6g", VKV_6, mistakes={"vᵀKv = Σᵢ Σⱼ vᵢ vⱼ Kᵢⱼ : les termes croisés comptent deux fois": VKV_ONCE_6, "vᵀKv = Σᵢ Σⱼ vᵢ vⱼ Kᵢⱼ : vérifie le signe de chaque produit vᵢ vⱼ": -VKV_6})

WBM_ANSWER {"decimals": 0, "element_hashes": ["4995914e7e4e0e63f4a7501784eed3f1e053ab63828bd84244c1d887377babaa", "c865b2fbf8dc7cf048036cfbf44b17c0a93918553f9c89e3f953baf4c1d07189"], "hash": "6511c6a3591494ccc64f75d32a0baf3ba71ae35c282001e565d2e30dfb219942", "id": "CP3.6a", "integer": true, "kind": "array", "mistakes": {"1333a2c5d74a977b442d7b3778e35475db9676ec089875f35fe52f5ded0f8477": "c'est le point du minimum de f sur l'ellipse"}, "shape": [2]}
📝 Ex CP3.6a : réponse enregistrée (array, 0 décimale).
WBM_ANSWER {"hash": "765b27fe8e32239562b8a6e4b46397c3fa4f80e94a8d80b4a024d2fa9119b559", "id": "CP3.6b", "kind": "int", "magnitude": 0, "mistakes": {"037eb4964ff5e14b925ebd8d02d9cddce3108680119272c0caeb8f94891af8ae": "c'est le minimum de f sur l'ellipse"}, "sign": 1}
📝 Ex CP3.6b : réponse enregistrée (int).
WBM_ANSWER {"hash": "00bb9d3045f077b342839d17fb5a76c4a565470256142baaefa328b852595f80", "id": "CP3.6c", "kind": "int", "magnitude": 0, "mistakes": {"5c4eec8384e049d0a2d344c50a013058ea5

**CP3.7 — Queue binomiale et borne de Chernoff, limite et poids exponentiels**

In [9]:
wbm.record("CP3.7a", TAIL_7, decimals=5, mistakes={"P(X ≥ 5) compte aussi le cas X = 5": TAIL_7 - pmf_7(5), "P(X ≥ 5) additionne P(X = k) pour tous les k ≥ 5, pas seulement k = 5": pmf_7(5)})
wbm.record("CP3.7b", N_7 * kl(A_7, P_7), decimals=3, mistakes={"la KL n'est pas symétrique : KL(a ‖ p), avec a en premier": N_7 * kl(P_7, A_7), "la question demande l'exposant n KL(a ‖ p), pas la borne": {math.exp(-N_7 * kl(A_7, P_7)), 0.015}, "la KL de la borne de Chernoff est en nats : logarithme népérien": N_7 * kl(A_7, P_7) / math.log(2)})
wbm.record("CP3.7c", math.exp(-2 * N_7 * (A_7 - P_7) ** 2), decimals=2, mistakes={"la borne de Hoeffding est e^(−2n(a − p)²), avec le facteur 2": math.exp(-N_7 * (A_7 - P_7) ** 2)})
wbm.record("CP3.7d", D_7, decimals=4, mistakes={"c'est la limite ; la question demande la valeur pour n = 45": LIMIT_7})
wbm.record("CP3.7e", GAP_7, decimals=4, mistakes={"n ln(1 + x/n) est-il au-dessus ou au-dessous de x ? Pense à ln(1 + u) ≤ u": -GAP_7, "c'est l'erreur relative ; la question demande l'écart des logarithmes": (D_7 - LIMIT_7) / LIMIT_7, "c'est le premier terme du développement, −x²/(2n) ; la question demande l'écart exact": -X_7 ** 2 / (2 * M_7)})
wbm.record("CP3.7f", weights_7([-ETA_7 * x for x in L_7]), decimals=4, mistakes={"les poids décroissent quand L grandit : e^(−η L)": weights_7([ETA_7 * x for x in L_7]), "n'oublie pas η dans l'exponentielle": weights_7([-x for x in L_7]), "normalise : la somme des poids vaut 1": [math.exp(-ETA_7 * (x - min(L_7))) for x in L_7], "les poids demandés sont normalisés : leur somme vaut 1": [math.exp(-ETA_7 * x) for x in L_7]})

WBM_ANSWER {"coarse_any": {"3": ["81b73317a92bf43d557604a6ca07fc21c122bb2afa5a75f2f466afdd3dc46214"]}, "decimals": 5, "hash": "2bc3c0494e062c7563992a129100986a7aa6d5090f6660144643c0e6568e5fe1", "hash_coarse": "594489b6e40fbc00b8ec1cbc29d0e3fa932f45cbc06a7e3a9d8cd0d6241f23d6", "id": "CP3.7a", "kind": "float", "magnitude": -3, "mistakes": {"016d35657cebd31568c9e7a925eb9940fa5ad42479d4f8602ab47837454c03e3": "P(X ≥ 5) compte aussi le cas X = 5", "f3732f4ebc1fc47ee74222e2852d4fd91dfc592c9dddc4ac7b906405cc0111c9": "P(X ≥ 5) additionne P(X = k) pour tous les k ≥ 5, pas seulement k = 5"}, "sign": 1}
📝 Ex CP3.7a : réponse enregistrée (float, 5 décimales).
WBM_ANSWER {"coarse_any": {"0": ["80633aa51b6d20b69e7ab604378d89f77286aaf03d70bd2c55b25a8d606aad24"]}, "coarser": {"1": ["5691b8100ebe1b43471af6116a917fee1c0a979baf760251283fd46ebdfa401b"]}, "decimals": 3, "hash": "97229149b4acd5e8b9604a9431552c010c5b4ecb08cb6851ccd60f5e1f3512bd", "hash_coarse": "a870c167ceccd72775634dd3a6952601550aa394e41bf4a

**CP3.9 — Corriger une copie : cinq erreurs (les lettres seulement)**

In [10]:
wbm.record("CP3.9a", "ACDFH", mistakes={"il manque une des lignes fausses : il y en a exactement cinq": {'ACDF', 'ACDH', 'ACFH', 'ADFH', 'CDFH'}, "une des lignes citées est juste : relis-les une par une, avec la règle qu'elle utilise": {'ABCDFH', 'ACDEFH', 'ACDFGH'}, "une ligne juste est citée à la place d'une ligne fausse": {'ABCDF', 'ABCDH', 'ABCFH', 'ABDFH', 'ACDEF', 'ACDEH', 'ACDFG', 'ACDGH', 'ACEFH', 'ACFGH', 'ADEFH', 'ADFGH', 'BCDFH', 'CDEFH', 'CDFGH'}, "ce sont les trois lignes justes ; la question demande les cinq lignes fausses": 'BEG'})

WBM_ANSWER {"hash": "523666483837513be4f64cb6ab61c53a1599064d52fa12914f3895b84b0be375", "id": "CP3.9a", "kind": "str", "mistakes": {"06df60fcb91e03cb1e6ffe2067735fee3f1532a90f3c1dfc2c505097f29444ae": "une des lignes citées est juste : relis-les une par une, avec la règle qu'elle utilise", "06f149839d20167a975fb3cbbab8a6dd7d7579bff30aa2ba4d737be8dc5ba5fe": "une ligne juste est citée à la place d'une ligne fausse", "0aa61aa89f424ef3dbf020ed58454506ece6d03f58a93bcb1fa0a79d1dd401ae": "une ligne juste est citée à la place d'une ligne fausse", "152cb79badd19dddadea18b97be58ebc1ad1a3dc00432f9c5fdf11f9d24004c4": "une ligne juste est citée à la place d'une ligne fausse", "23c01af07effffb0b7f1150162a08fb23b0a543a4e3ad6d391099338776f0813": "une ligne juste est citée à la place d'une ligne fausse", "3d3a7600aca34cb2a39d945a6e115f99c50cf6039121a278df4d7fd59ff067e9": "une ligne juste est citée à la place d'une ligne fausse", "4790f9d6eb4b50501035fef0aa021c7b12505eed9225fab585e73d02797dbdb5": "une li

**CP3.10 — Problème de synthèse : des données centrées, leurs axes principaux, un modèle sur les scores**

In [11]:
wbm.record("CP3.10a", whole(MEAN_10))
wbm.record("CP3.10b", whole(T_10[0]), mistakes={"le score se calcule sur l'exemple centré, x₁ − x̄": scores_10(X_10[0], center=False), "les axes sont les vecteurs donnés divisés par 21": whole([21 * t for t in T_10[0]])})
wbm.record("CP3.10c", [ss / (N_10 - 1) for ss in SS_10], decimals=1, mistakes={"l'énoncé demande la variance avec n − 1": [ss / N_10 for ss in SS_10], "ce sont les variances des features ; la question porte sur les scores": [sum((x[j] - MEAN_10[j]) ** 2 for x in X_10) / (N_10 - 1) for j in range(3)]})
wbm.record("CP3.10d", (SS_10[0] + SS_10[1]) / sum(SS_10), decimals=3, mistakes={"deux axes, pas un seul": SS_10[0] / sum(SS_10), "c'est la part perdue ; la question demande la part gardée": SS_10[2] / sum(SS_10)})
wbm.record("CP3.10e", W_10[:2], decimals=3, mistakes={"les équations normales divisent t_j · y par Σ t_j², pas par la variance": [ty / (ss / (N_10 - 1)) for ty, ss in zip(TY_10[:2], SS_10[:2])]})
wbm.record("CP3.10f", R2_10[1], decimals=3, mistakes={"deux axes, pas un seul": R2_10[0], "c'est le R² à trois axes ; la question demande celui à deux axes": R2_10[2], "la somme des carrés totale se calcule sur y − ȳ": R2_10[1] * SCT_10 / sum(y * y for y in Y_10)})
wbm.record("CP3.10g", whole(W_10[2]), mistakes={"les équations normales divisent t₃ · y par Σ t₃², pas par la variance": whole(TY_10[2] / (SS_10[2] / (N_10 - 1)))})
wbm.record("CP3.10h", R2_10[2], decimals=3, mistakes={"c'est le R² à deux axes ; la question demande celui à trois axes": R2_10[1]})
wbm.record("CP3.10i", whole([predict_10(NEW_10, 2), predict_10(NEW_10, 3)]), mistakes={"le nouvel exemple se centre avec la moyenne d'entraînement avant de calculer ses scores": [predict_10(NEW_10, k, center=False) for k in (2, 3)], "l'ordre : la prédiction à deux axes, puis celle à trois axes": whole([predict_10(NEW_10, 3), predict_10(NEW_10, 2)]), "le nouvel exemple se centre avec la moyenne d'entraînement, pas avec ses propres valeurs": whole([YBAR_10, YBAR_10]), "le modèle prédit ȳ plus les termes en t : as-tu ajouté ȳ ?": whole([predict_10(NEW_10, k) - YBAR_10 for k in (2, 3)])})

WBM_ANSWER {"decimals": 0, "element_hashes": ["0c7c118821336710916d2bbe2cd1cdcc9b73326f063e5f08ed4709bc48ae2ee6", "b7d65d722a1e249be63b35061838341614da6fb0f8f97c9da49b88fbb830d722", "26838d0899df2d5335cc9d097b3a2b0988bad168ac414ae690ff43dd625be537"], "hash": "23f280bce9c5bb18a367a628e1145ebd5f701b3881ff31f8bb9c4ee8b25418cb", "id": "CP3.10a", "integer": true, "kind": "array", "shape": [3]}
📝 Ex CP3.10a : réponse enregistrée (array, 0 décimale).
WBM_ANSWER {"decimals": 0, "element_hashes": ["5468e6e162fcedb641889b546d266f6f6949fbd314b60779e7be5a9fe225d52e", "db248f105e73daa5451c5cea3e6648f31c5da71c6488198daa03bed982fec0f7", "22e931c4bdbcc2676a05a0975ccaccd5bcb97db9d5ed82dd836e7eb4ba3073ec"], "hash": "7336606228523f5078624a2c94dd1989c41a5c3eb89a5ec02a64c409e252f209", "id": "CP3.10b", "integer": true, "kind": "array", "mistake_decimals": 2, "mistakes": {"ad8693c74f8a6de9cc89dba4ce04a4f2bb9d59d8725c8b356af4777fbf6d4b20": "les axes sont les vecteurs donnés divisés par 21", "e2bc45a509f182cfc